# Day 40 — Practice 1: Pindahkan File dari Local ke HDFS

**Tujuan:** Memahami cara kerja HDFS sebagai Distributed Storage (Data Lake layer).

Alur:
```
Local File System  →  HDFS (Data Lake)
```

**Konsep yang dipraktikkan:**
- HDFS CLI commands (`hdfs dfs`)
- Block storage & replication
- Manajemen direktori di HDFS
- Upload / download file
- Cek metadata file di HDFS

## Setup: Koneksi ke HDFS via Python

Kita akan pakai `subprocess` untuk menjalankan `hdfs dfs` command dari dalam notebook,
sekaligus module `hdfs` (WebHDFS client) untuk operasi yang lebih Python-native.

In [ ]:
import subprocess
import os

# Helper: jalankan hdfs dfs command
def hdfs(cmd: str, capture=True):
    """
    Jalankan hdfs dfs command di dalam container namenode.
    Contoh: hdfs('ls /data/raw')
    """
    full_cmd = f'docker exec hadoop-namenode hdfs dfs -{cmd}'
    result = subprocess.run(full_cmd, shell=True, capture_output=capture, text=True)
    if result.returncode != 0 and result.stderr:
        print(f'[ERROR] {result.stderr.strip()}')
    if capture and result.stdout:
        print(result.stdout)
    return result

def hdfs_admin(cmd: str):
    """Jalankan hdfs dfsadmin command"""
    full_cmd = f'docker exec hadoop-namenode hdfs dfsadmin -{cmd}'
    result = subprocess.run(full_cmd, shell=True, capture_output=True, text=True)
    print(result.stdout)
    return result

# Cek HDFS bisa diakses
print('=== Cek HDFS Status ===')
hdfs('ls /')

## 1. Buat Struktur Direktori di HDFS

Kita ikuti konvensi **Data Lake layering**:
```
/datalake
  /raw        ← raw files (csv, json, parquet dari sumber)
  /staging    ← data yang sedang diproses
  /curated    ← data bersih siap dikonsumsi
```

In [ ]:
# Buat folder datalake structure di HDFS
for path in ['/datalake/raw', '/datalake/staging', '/datalake/curated']:
    hdfs(f'mkdir -p {path}')
    print(f'Created: {path}')

print('\n=== Struktur HDFS setelah dibuat ===')
hdfs('ls -R /datalake')

## 2. Buat File Sample di Local

Simulasikan file yang ada di local machine sebelum dipindahkan ke HDFS.

In [ ]:
import csv, os

# Buat folder local untuk menyimpan file sementara
os.makedirs('/tmp/local_data', exist_ok=True)

# ---- File 1: employees.csv ----
employees = [
    ['emp_id', 'name', 'department', 'salary', 'hire_date'],
    [1, 'Andi Pratama', 'Engineering', 8500000, '2022-03-01'],
    [2, 'Siti Rahayu', 'Marketing', 7200000, '2021-07-15'],
    [3, 'Budi Santoso', 'Engineering', 9100000, '2020-11-20'],
    [4, 'Dewi Kusuma', 'Finance', 8000000, '2023-01-10'],
    [5, 'Reza Firmansyah', 'Engineering', 9500000, '2019-05-08'],
]
with open('/tmp/local_data/employees.csv', 'w', newline='') as f:
    csv.writer(f).writerows(employees)

# ---- File 2: sales_summary.csv ----
sales = [
    ['date', 'region', 'product', 'qty', 'revenue'],
    ['2024-01-01', 'Jakarta', 'Laptop', 10, 150000000],
    ['2024-01-01', 'Surabaya', 'Monitor', 20, 60000000],
    ['2024-01-02', 'Jakarta', 'Keyboard', 50, 15000000],
    ['2024-01-02', 'Bandung', 'Laptop', 5, 75000000],
    ['2024-01-03', 'Jakarta', 'Mouse', 100, 20000000],
]
with open('/tmp/local_data/sales_summary.csv', 'w', newline='') as f:
    csv.writer(f).writerows(sales)

# ---- File 3: config.json ----
import json
config = {'env': 'production', 'version': '1.0', 'tags': ['bigdata', 'datalake']}
with open('/tmp/local_data/config.json', 'w') as f:
    json.dump(config, f, indent=2)

print('=== File yang sudah dibuat di local ===')
for f in os.listdir('/tmp/local_data'):
    size = os.path.getsize(f'/tmp/local_data/{f}')
    print(f'  {f:30s}  {size} bytes')

## 3. Upload File dari Local ke HDFS

Dua cara upload:
- `hdfs dfs -put` → copy dari local ke HDFS (file lokal tetap ada)
- `hdfs dfs -moveFromLocal` → pindah dari local ke HDFS (file lokal dihapus)

In [ ]:
# Copy file dari container Jupyter ke container NameNode dulu,
# lalu upload ke HDFS dari NameNode

def upload_to_hdfs(local_path: str, hdfs_path: str):
    """
    Upload file dari local ke HDFS.
    1. Copy file ke dalam container namenode via docker cp
    2. Upload ke HDFS dari dalam container
    """
    filename = os.path.basename(local_path)
    tmp_in_container = f'/tmp/{filename}'
    
    # Step 1: copy ke container
    subprocess.run(f'docker cp {local_path} hadoop-namenode:{tmp_in_container}',
                   shell=True, check=True)
    
    # Step 2: upload ke HDFS
    result = subprocess.run(
        f'docker exec hadoop-namenode hdfs dfs -put -f {tmp_in_container} {hdfs_path}',
        shell=True, capture_output=True, text=True
    )
    if result.returncode == 0:
        print(f'✓ Uploaded: {filename} → {hdfs_path}/{filename}')
    else:
        print(f'✗ Failed: {result.stderr}')

# Upload semua file ke /datalake/raw
for fname in ['employees.csv', 'sales_summary.csv', 'config.json']:
    upload_to_hdfs(f'/tmp/local_data/{fname}', '/datalake/raw')

print('\n=== Isi /datalake/raw setelah upload ===')
hdfs('ls -l /datalake/raw')

## 4. Verifikasi File di HDFS

In [ ]:
# Baca isi file langsung dari HDFS
print('=== Isi employees.csv di HDFS ===')
hdfs('cat /datalake/raw/employees.csv')

print('\n=== Isi sales_summary.csv di HDFS ===')
hdfs('cat /datalake/raw/sales_summary.csv')

In [ ]:
# Cek detail block info setiap file
print('=== Block info: employees.csv ===')
subprocess.run(
    'docker exec hadoop-namenode hdfs fsck /datalake/raw/employees.csv -files -blocks -locations',
    shell=True
)

print('\n=== Disk usage /datalake ===')
hdfs('du -h /datalake')

## 5. Operasi File Lainnya di HDFS

In [ ]:
# Copy file di dalam HDFS
hdfs('cp /datalake/raw/employees.csv /datalake/staging/employees_copy.csv')
print('Copy ke staging:')
hdfs('ls /datalake/staging')

# Move / rename file
hdfs('mv /datalake/staging/employees_copy.csv /datalake/staging/employees_v1.csv')
print('\nSetelah rename:')
hdfs('ls /datalake/staging')

# Hapus file
hdfs('rm /datalake/staging/employees_v1.csv')
print('\nSetelah hapus:')
hdfs('ls /datalake/staging')

In [ ]:
# Download file dari HDFS ke local
subprocess.run(
    'docker exec hadoop-namenode hdfs dfs -get /datalake/raw/sales_summary.csv /tmp/downloaded_sales.csv',
    shell=True
)
subprocess.run('docker cp hadoop-namenode:/tmp/downloaded_sales.csv /tmp/local_data/', shell=True)
print('File berhasil di-download ke local:')
print(open('/tmp/local_data/downloaded_sales.csv').read())

## 6. Cek HDFS Cluster Health

In [ ]:
# Laporan cluster
print('=== HDFS Cluster Report ===')
hdfs_admin('report')

print('\n=== Total disk usage ===')
hdfs('df -h /')

## 7. MapReduce dengan Hadoop Streaming

**Konsep:** MapReduce adalah model pemrograman paralel untuk memproses data besar:
1. **Map** — Setiap worker baca sebagian data, output pasangan `(key, value)`
2. **Shuffle** — Sistem otomatis sort dan kelompokkan semua value per key
3. **Reduce** — Setiap reducer terima satu key + semua value-nya, output hasil akhir

```
Input                 Map                   Shuffle             Reduce            Output
"spark hadoop"   →   (spark,1)          →  (hadoop,[1,1,1]) → (hadoop,3)   →  hadoop  3
"hadoop hive"        (hadoop,1)             (hive,[1,1])       (hive,2)         hive    2
"hadoop spark"       (hive,1)               (spark,[1,1])      (spark,2)        spark   2
                     (hadoop,1)
                     (spark,1)
```

Kita pakai **Hadoop Streaming** — fitur bawaan Hadoop yang memungkinkan mapper dan reducer
ditulis dalam bahasa apapun (Python, Bash, dll) melalui stdin/stdout.

### 7.1 Siapkan Data Input di HDFS

In [ ]:
# Buat file teks input untuk MapReduce
text_lines = [
    'apache spark is a fast and general purpose cluster computing system',
    'apache hadoop is a framework for distributed storage and processing',
    'apache hive is a data warehouse built on top of hadoop',
    'spark provides an interface for programming entire clusters',
    'hadoop uses hdfs for distributed storage across many nodes',
    'hive translates sql queries into mapreduce jobs on hadoop',
    'spark is much faster than mapreduce because it uses memory',
    'hadoop mapreduce writes intermediate results to disk between steps',
    'spark keeps data in memory making iterative algorithms very fast',
    'apache kafka is a distributed streaming platform for hadoop ecosystem',
]

with open('/tmp/local_data/wordcount_input.txt', 'w') as f:
    f.write('\n'.join(text_lines))

# Upload ke HDFS
subprocess.run('docker cp /tmp/local_data/wordcount_input.txt hadoop-namenode:/tmp/', shell=True)
subprocess.run('docker exec hadoop-namenode hdfs dfs -mkdir -p /datalake/raw/mapreduce_input', shell=True)
subprocess.run('docker exec hadoop-namenode hdfs dfs -put -f /tmp/wordcount_input.txt /datalake/raw/mapreduce_input/', shell=True)

print('Input file tersimpan di HDFS:')
subprocess.run('docker exec hadoop-namenode hdfs dfs -cat /datalake/raw/mapreduce_input/wordcount_input.txt', shell=True)

### 7.2 Buat Mapper Script (Python)

Mapper membaca baris dari stdin, memecah per kata, dan output `word\t1` ke stdout.

In [ ]:
mapper_lines = [
    '#!/usr/bin/env python3',
    '# mapper.py — baca baris dari stdin, output (word, 1) per kata',
    'import sys',
    '',
    'for line in sys.stdin:',
    '    line = line.strip().lower()',
    '    words = line.split()',
    '    for word in words:',
    '        # Format output: key TAB value',
    '        print(f"{word}\\t1")',
]

with open('/tmp/local_data/mapper.py', 'w') as f:
    f.write('\n'.join(mapper_lines))

print('mapper.py:')
print('\n'.join(mapper_lines))

# Test mapper lokal (simulasi)
print('\n=== Output mapper untuk 1 baris ===')
sample = 'spark hadoop spark hive hadoop spark'
for word in sample.strip().lower().split():
    print(f'{word}\t1')

### 7.3 Buat Reducer Script (Python)

Reducer menerima input yang sudah di-sort per key dari Shuffle phase,
lalu menjumlahkan semua count untuk setiap kata yang sama.

In [ ]:
reducer_lines = [
    '#!/usr/bin/env python3',
    '# reducer.py — terima (word, 1) yang sudah di-sort, jumlahkan per kata',
    'import sys',
    '',
    'current_word  = None',
    'current_count = 0',
    '',
    'for line in sys.stdin:',
    '    line = line.strip()',
    '    if not line:',
    '        continue',
    '    word, count = line.split("\\t", 1)',
    '    count = int(count)',
    '',
    '    if word == current_word:',
    '        # Key sama -> akumulasi',
    '        current_count += count',
    '    else:',
    '        # Key baru -> output key sebelumnya',
    '        if current_word is not None:',
    '            print(f"{current_word}\\t{current_count}")',
    '        current_word  = word',
    '        current_count = count',
    '',
    '# Output kata terakhir',
    'if current_word is not None:',
    '    print(f"{current_word}\\t{current_count}")',
]

with open('/tmp/local_data/reducer.py', 'w') as f:
    f.write('\n'.join(reducer_lines))

print('reducer.py:')
print('\n'.join(reducer_lines))

# Test pipeline lokal: mapper | sort | reducer (simulasi MapReduce)
print('\n=== Test pipeline lokal (simulasi MapReduce) ===')
result = subprocess.run(
    'cat /tmp/local_data/wordcount_input.txt'
    ' | python3 /tmp/local_data/mapper.py'
    ' | sort'
    ' | python3 /tmp/local_data/reducer.py'
    ' | sort -t$"\t" -k2 -rn'
    ' | head -15',
    shell=True, capture_output=True, text=True
)
print(result.stdout)

### 7.4 Jalankan MapReduce Job di Hadoop Cluster

Setelah diuji lokal, sekarang jalankan di Hadoop cluster menggunakan **Hadoop Streaming JAR**.
Hadoop Streaming akan:
1. Bagi input file dari HDFS ke tiap mapper (satu task per block)
2. Kumpulkan output mapper, sort by key (Shuffle phase)
3. Kirim ke reducer, simpan hasil ke HDFS

In [ ]:
# Copy scripts ke container
for script in ['mapper.py', 'reducer.py']:
    subprocess.run(f'docker cp /tmp/local_data/{script} hadoop-namenode:/tmp/{script}', shell=True)
    subprocess.run(f'docker exec hadoop-namenode chmod +x /tmp/{script}', shell=True)
print('Scripts ter-copy ke container')

# Hapus output lama
subprocess.run('docker exec hadoop-namenode hdfs dfs -rm -r -f /datalake/raw/mapreduce_output', shell=True)

# Cari path hadoop-streaming jar
find_jar = subprocess.run(
    'docker exec hadoop-namenode find /opt/hadoop -name "hadoop-streaming-*.jar" 2>/dev/null | head -1',
    shell=True, capture_output=True, text=True
)
streaming_jar = find_jar.stdout.strip()
print(f'Streaming JAR: {streaming_jar}')

# Jalankan MapReduce job
print('\nMenjalankan MapReduce job...')
mr_cmd = (
    f'docker exec hadoop-namenode '
    f'hadoop jar {streaming_jar} '
    f'-files /tmp/mapper.py,/tmp/reducer.py '
    f'-mapper "python3 mapper.py" '
    f'-reducer "python3 reducer.py" '
    f'-input  /datalake/raw/mapreduce_input/wordcount_input.txt '
    f'-output /datalake/raw/mapreduce_output'
)

result = subprocess.run(mr_cmd, shell=True, capture_output=True, text=True)

# Tampilkan bagian akhir log (progress info)
log = result.stderr if result.stderr else result.stdout
relevant = [l for l in log.splitlines() if any(k in l for k in ['map', 'reduce', 'INFO', 'ERROR', 'completed', 'failed'])]
print('\n'.join(relevant[-20:]))

if result.returncode == 0:
    print('\n✅ MapReduce job selesai!')
else:
    print(f'\n❌ Job gagal. Stderr:\n{result.stderr[-1500:]}')

### 7.5 Lihat Hasil Output MapReduce

In [ ]:
# File output yang dihasilkan
print('=== Output files di HDFS ===')
subprocess.run('docker exec hadoop-namenode hdfs dfs -ls /datalake/raw/mapreduce_output', shell=True)

# Baca dan sort hasil (descending by count)
print('\n=== Hasil Word Count (Top 20 kata) ===')
result = subprocess.run(
    'docker exec hadoop-namenode bash -c '
    '"hdfs dfs -cat /datalake/raw/mapreduce_output/part-* | sort -t$\'\\t\' -k2 -rn | head -20"',
    shell=True, capture_output=True, text=True
)
# Pretty print
for line in result.stdout.strip().splitlines():
    parts = line.split('\t')
    if len(parts) == 2:
        word, count = parts
        bar = chr(9608) * int(count)
        print(f'  {word:15s} {count:>3s}  {bar}')

# Hitung total kata unik
result2 = subprocess.run(
    'docker exec hadoop-namenode bash -c "hdfs dfs -cat /datalake/raw/mapreduce_output/part-* | wc -l"',
    shell=True, capture_output=True, text=True
)
print(f'\nTotal kata unik: {result2.stdout.strip()}')

### 7.6 Baca Output MapReduce dengan Spark

Output MapReduce (plain text TSV di HDFS) bisa langsung dibaca Spark.
Inilah kekuatan ekosistem Hadoop: semua tool berbagi satu storage yang sama.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

spark = SparkSession.builder \
    .master('local[*]') \
    .appName('Read-MapReduce-Output') \
    .getOrCreate()

# Baca output MapReduce: format word TAB count
schema = StructType([
    StructField('word',  StringType(),  True),
    StructField('count', IntegerType(), True),
])

df_wc = spark.read \
    .option('sep', '\t') \
    .option('header', False) \
    .schema(schema) \
    .csv('hdfs://namenode:9000/datalake/raw/mapreduce_output/part-*')

print(f'Total kata unik: {df_wc.count()}')
print('\n=== Top 20 kata (via Spark) ===')
df_wc.orderBy(F.desc('count')).show(20)

print('=== Kata yang muncul > 2 kali ===')
df_wc.filter(F.col('count') > 2).orderBy(F.desc('count')).show()

### 7.7 Perbandingan: MapReduce vs Spark RDD

Operasi word count yang sama, dua cara yang berbeda:

In [ ]:
import time

sc = spark.sparkContext

# Spark RDD Word Count
t0 = time.time()

rdd_text = sc.textFile('hdfs://namenode:9000/datalake/raw/mapreduce_input/wordcount_input.txt')

spark_wc = rdd_text \
    .flatMap(lambda line: line.strip().lower().split()) \
    .map(lambda word: (word, 1)) \
    .reduceByKey(lambda a, b: a + b) \
    .sortBy(lambda x: x[1], ascending=False)

results = spark_wc.collect()
t_spark = time.time() - t0

print(f'Spark RDD selesai dalam {t_spark:.2f}s')
print('\nTop 15 kata (Spark RDD):')
for word, count in results[:15]:
    bar = chr(9608) * count
    print(f'  {word:15s} {count:3d}  {bar}')

print('\n--- Kesimpulan Perbandingan ---')
rows = [
    ('Storage intermediate', 'Tulis ke disk (HDFS)', 'Di memori (RAM)'),
    ('Kecepatan',            'Lambat (I/O disk)',    'Cepat (100x lebih cepat)'),
    ('Fault tolerance',      'Per task ke disk',     'Lineage-based DAG'),
    ('API',                  'stdin/stdout (apapun)','Python/Scala/Java/R/SQL'),
    ('Use case',             'Batch besar, legacy',  'Batch + streaming + ML'),
    ('Hadoop Streaming',     'Ya (native)',           'Tidak perlu'),
]
print(f'{"Aspek":25s} {"MapReduce":30s} {"Spark"}')
print('-' * 80)
for aspect, mr, sp in rows:
    print(f'{aspect:25s} {mr:30s} {sp}')

## Summary

Yang sudah dipraktikkan:
- ✅ Membuat struktur direktori Data Lake di HDFS (`/datalake/raw`, `staging`, `curated`)
- ✅ Upload file dari local ke HDFS dengan `hdfs dfs -put`
- ✅ Verifikasi isi file dengan `hdfs dfs -cat`
- ✅ Cek block info dan replication factor
- ✅ Copy, move, rename, dan hapus file di HDFS
- ✅ Download file dari HDFS kembali ke local
- ✅ Monitoring cluster health
- ✅ **MapReduce dengan Hadoop Streaming** — tulis mapper & reducer Python, jalankan di Hadoop cluster
- ✅ Baca output MapReduce dari HDFS menggunakan Spark
- ✅ Perbandingan MapReduce vs Spark RDD

**Next →** `02_adventureworks_to_hdfs.ipynb` — Extract AdventureWorks dari PostgreSQL dan simpan ke HDFS sebagai Parquet.